In [ ]:
!pip -q install feast==0.64.0 pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 72.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.1/64.1 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 68.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 56.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.9/531.9 kB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 4.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.4.0 requires tenacity<1

In [ ]:
import feast

print("Feast version:", feast.__version__)

Feast version: 0.64.0


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
import pandas as pd

df = pd.read_csv("/content/D31 CSE dataset.csv")

print(df.head())
print(df.shape)

   sample_id         degree_domain         curriculum_skills  \
0          1  Computer Engineering            Java, OOP, SQL   
1          2         Cybersecurity          Git, Python, SQL   
2          3      Computer Science            Java, OOP, SQL   
3          4         Cybersecurity     CSS, HTML, JavaScript   
4          5  Computer Engineering  Data Mining, Python, SQL   

                         student_skills             industry_required_skills  \
0                             Java, SQL            CI/CD, Kubernetes, Docker   
1     Cloud Computing, Git, Python, SQL               AWS, Docker, Terraform   
2                        Java, OOP, SQL            IAM, SIEM, Cloud Security   
3     Cloud Computing, HTML, JavaScript          Docker, Kubernetes, Jenkins   
4  Data Mining, JavaScript, Python, SQL  Cloud Databases, Docker, PostgreSQL   

          skill_gap gap_level           target_role  \
0  CI/CD automation      High       DevOps Engineer   
1      Advanced SQL    M

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
print(df.columns)
print(df.info())
print(df.isnull().sum())

Index(['sample_id', 'degree_domain', 'curriculum_skills', 'student_skills',
       'industry_required_skills', 'skill_gap', 'gap_level', 'target_role',
       'curriculum_industry_alignment', 'student_skill_score',
       'industry_demand_score', 'missing_skill_count',
       'training_hours_recommended'],
      dtype='object')
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 13 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   sample_id                      2000 non-null   int64 
 1   degree_domain                  2000 non-null   object
 2   curriculum_skills              2000 non-null   object
 3   student_skills                 2000 non-null   object
 4   industry_required_skills       2000 non-null   object
 5   skill_gap                      2000 non-null   object
 6   gap_level                      2000 non-null   object
 7   target_role                

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
import pandas as pd

df["sample_id"] = df["sample_id"].astype("int64")

df["curriculum_industry_alignment"] = df["curriculum_industry_alignment"].fillna(
    df["curriculum_industry_alignment"].median()
)

df["student_skill_score"] = df["student_skill_score"].fillna(
    df["student_skill_score"].median()
)

df["industry_demand_score"] = df["industry_demand_score"].fillna(
    df["industry_demand_score"].median()
)

df["missing_skill_count"] = df["missing_skill_count"].fillna(
    df["missing_skill_count"].median()
)

df["training_hours_recommended"] = df["training_hours_recommended"].fillna(
    df["training_hours_recommended"].median()
)

gap_mapping = {
    "Low": 0,
    "Medium": 1,
    "High": 2
}

df["gap_level_encoded"] = df["gap_level"].map(gap_mapping).fillna(0).astype("int64")

df["alignment_score"] = df["curriculum_industry_alignment"].astype("float32")
df["student_score"] = df["student_skill_score"].astype("float32")
df["industry_score"] = df["industry_demand_score"].astype("float32")
df["missing_skills"] = df["missing_skill_count"].astype("int64")
df["training_hours"] = df["training_hours_recommended"].astype("float32")

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
base_time = pd.Timestamp(
    "2025-01-01",
    tz="UTC"
)

df["event_timestamp"] = (
    base_time +
    pd.to_timedelta(
        df["sample_id"],
        unit="s"
    )
)

df["created_timestamp"] = (
    df["event_timestamp"] +
    pd.Timedelta(seconds=1)
)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
feature_df = df[
    [
        "sample_id",
        "event_timestamp",
        "created_timestamp",
        "alignment_score",
        "student_score",
        "industry_score",
        "missing_skills",
        "training_hours",
        "gap_level_encoded"
    ]
].copy()

display(feature_df.head())

,sample_id,event_timestamp,created_timestamp,alignment_score,student_score,industry_score,missing_skills,training_hours,gap_level_encoded
0,1,2025-01-01 00:00:01+00:00,2025-01-01 00:00:02+00:00,0.0,1.0,3.0,3,32.0,2
1,2,2025-01-01 00:00:02+00:00,2025-01-01 00:00:03+00:00,5.0,4.0,2.0,3,21.0,1
2,3,2025-01-01 00:00:03+00:00,2025-01-01 00:00:04+00:00,1.0,8.0,4.0,10,23.0,2
3,4,2025-01-01 00:00:04+00:00,2025-01-01 00:00:05+00:00,5.0,5.0,3.0,10,17.0,2
4,5,2025-01-01 00:00:05+00:00,2025-01-01 00:00:06+00:00,10.0,8.0,3.0,10,20.0,1


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
label_df = df[
    [
        "sample_id",
        "event_timestamp",
        "gap_level_encoded"
    ]
].copy()

label_df = label_df.rename(
    columns={
        "gap_level_encoded": "gap_level"
    }
)

display(label_df.head())

,sample_id,event_timestamp,gap_level
0,1,2025-01-01 00:00:01+00:00,2
1,2,2025-01-01 00:00:02+00:00,1
2,3,2025-01-01 00:00:03+00:00,2
3,4,2025-01-01 00:00:04+00:00,2
4,5,2025-01-01 00:00:05+00:00,1


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
import os

repo_path = "/content/cse_feast"

os.makedirs(
    f"{repo_path}/data",
    exist_ok=True
)

In [ ]:
feature_df.to_parquet(
    f"{repo_path}/data/cse_features.parquet",
    index=False
)

In [ ]:
feature_store_yaml = """
project: cse_skill_gap_project

registry: data/registry.db

provider: local

offline_store:
  type: file

online_store:
  type: sqlite
  path: data/online_store.db
"""

with open(
    f"{repo_path}/feature_store.yaml",
    "w"
) as f:
    f.write(feature_store_yaml)

In [ ]:
feature_definition = '''
from datetime import timedelta

from feast import (
    Entity,
    FeatureView,
    FeatureService,
    Field,
    FileSource
)

from feast.types import (
    Float32,
    Int64
)

sample = Entity(
    name="sample",
    join_keys=["sample_id"],
    description="CSE curriculum industry skill gap sample"
)

cse_source = FileSource(
    name="cse_source",
    path="data/cse_features.parquet",
    timestamp_field="event_timestamp",
    created_timestamp_column="created_timestamp"
)

cse_feature_view = FeatureView(
    name="cse_skill_features",
    entities=[sample],

    ttl=timedelta(days=50000),

    schema=[
        Field(name="alignment_score", dtype=Float32),
        Field(name="student_score", dtype=Float32),
        Field(name="industry_score", dtype=Float32),
        Field(name="missing_skills", dtype=Int64),
        Field(name="training_hours", dtype=Float32),
        Field(name="gap_level_encoded", dtype=Int64)
    ],

    source=cse_source,

    online=True
)

cse_feature_service = FeatureService(
    name="cse_skill_gap_service",
    features=[
        cse_feature_view
    ]
)
'''

with open(
    f"{repo_path}/features.py",
    "w"
) as f:
    f.write(feature_definition)

In [ ]:
!find /content/cse_feast -maxdepth 2 -type f

/content/cse_feast/feature_store.yaml
/content/cse_feast/features.py
/content/cse_feast/data/cse_features.parquet


In [ ]:
%cd /content/cse_feast

/content/cse_feast


In [ ]:
!feast apply

/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:64: PyparsingDeprecationWarning: 'oneOf' deprecated - use 'one_of'
  prop = Group((name + Suppress("=") + comma_separated(value)) | oneOf(_CONSTANTS))
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:85: PyparsingDeprecationWarning: 'parseString' deprecated - use 'parse_string'
  parse = parser.parseString(pattern)
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:89: PyparsingDeprecationWarning: 'resetCache' deprecated - use 'reset_cache'
  parser.resetCache()
/usr/local/lib/python3.12/dist-packages/matplotlib/_mathtext.py:45: PyparsingDeprecationWarning: 'enablePackrat' deprecated - use 'enable_packrat'
  ParserElement.enablePackrat()
In /usr/local/lib/python3.12/dist-packages/matplotlib/mpl-data/stylelib/classic.mplstyle: 'parseString' deprecated - use 'parse_string'
In /usr/local/lib/python3.12/dist-packages/matplotlib/mpl-data/stylelib/classic.mplstyle: 'reset

In [ ]:
!feast entities list

/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:64: PyparsingDeprecationWarning: 'oneOf' deprecated - use 'one_of'
  prop = Group((name + Suppress("=") + comma_separated(value)) | oneOf(_CONSTANTS))
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:85: PyparsingDeprecationWarning: 'parseString' deprecated - use 'parse_string'
  parse = parser.parseString(pattern)
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:89: PyparsingDeprecationWarning: 'resetCache' deprecated - use 'reset_cache'
  parser.resetCache()
/usr/local/lib/python3.12/dist-packages/matplotlib/_mathtext.py:45: PyparsingDeprecationWarning: 'enablePackrat' deprecated - use 'enable_packrat'
  ParserElement.enablePackrat()
In /usr/local/lib/python3.12/dist-packages/matplotlib/mpl-data/stylelib/classic.mplstyle: 'parseString' deprecated - use 'parse_string'
In /usr/local/lib/python3.12/dist-packages/matplotlib/mpl-data/stylelib/classic.mplstyle: 'reset

In [ ]:
!feast feature-views list

/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:64: PyparsingDeprecationWarning: 'oneOf' deprecated - use 'one_of'
  prop = Group((name + Suppress("=") + comma_separated(value)) | oneOf(_CONSTANTS))
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:85: PyparsingDeprecationWarning: 'parseString' deprecated - use 'parse_string'
  parse = parser.parseString(pattern)
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:89: PyparsingDeprecationWarning: 'resetCache' deprecated - use 'reset_cache'
  parser.resetCache()
/usr/local/lib/python3.12/dist-packages/matplotlib/_mathtext.py:45: PyparsingDeprecationWarning: 'enablePackrat' deprecated - use 'enable_packrat'
  ParserElement.enablePackrat()
In /usr/local/lib/python3.12/dist-packages/matplotlib/mpl-data/stylelib/classic.mplstyle: 'parseString' deprecated - use 'parse_string'
In /usr/local/lib/python3.12/dist-packages/matplotlib/mpl-data/stylelib/classic.mplstyle: 'reset

In [ ]:
from feast import FeatureStore

store = FeatureStore(
    repo_path="/content/cse_feast"
)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
feature_service = store.get_feature_service(
    "cse_skill_gap_service"
)

In [ ]:
entity_df = label_df.copy()

display(entity_df.head())

,sample_id,event_timestamp,gap_level
0,1,2025-01-01 00:00:01+00:00,2
1,2,2025-01-01 00:00:02+00:00,1
2,3,2025-01-01 00:00:03+00:00,2
3,4,2025-01-01 00:00:04+00:00,2
4,5,2025-01-01 00:00:05+00:00,1


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
training_data = store.get_historical_features(
    entity_df=entity_df,
    features=feature_service
).to_df()

display(training_data.head())

,sample_id,event_timestamp,gap_level,alignment_score,student_score,industry_score,missing_skills,training_hours,gap_level_encoded
0,1,2025-01-01 00:00:01+00:00,2,0.0,1.0,3.0,3,32.0,2
1,2,2025-01-01 00:00:02+00:00,1,5.0,4.0,2.0,3,21.0,1
2,3,2025-01-01 00:00:03+00:00,2,1.0,8.0,4.0,10,23.0,2
3,4,2025-01-01 00:00:04+00:00,2,5.0,5.0,3.0,10,17.0,2
4,5,2025-01-01 00:00:05+00:00,1,10.0,8.0,3.0,10,20.0,1


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
feature_columns = [
    "alignment_score",
    "student_score",
    "industry_score",
    "missing_skills",
    "training_hours"
]

In [ ]:
X = training_data[feature_columns]

y = training_data["gap_level"]

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier(
    max_depth=4,
    random_state=42
)

model.fit(
    X_train,
    y_train
)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


DecisionTreeClassifier(max_depth=4, random_state=42)

In [ ]:
predictions = model.predict(X_test)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(
    y_test,
    predictions
)

print(
    "Accuracy:",
    round(accuracy * 100, 2),
    "%"
)

Accuracy: 42.75 %


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test,
        predictions
    )
)

              precision    recall  f1-score   support

           0       0.33      0.01      0.02        88
           1       0.45      0.85      0.58       179
           2       0.32      0.14      0.19       133

    accuracy                           0.43       400
   macro avg       0.37      0.33      0.27       400
weighted avg       0.38      0.43      0.33       400



/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
%cd /content/cse_feast

!feast materialize \
    2025-01-01T00:00:00 \
    2025-01-02T00:00:00

/content/cse_feast
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:64: PyparsingDeprecationWarning: 'oneOf' deprecated - use 'one_of'
  prop = Group((name + Suppress("=") + comma_separated(value)) | oneOf(_CONSTANTS))
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:85: PyparsingDeprecationWarning: 'parseString' deprecated - use 'parse_string'
  parse = parser.parseString(pattern)
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:89: PyparsingDeprecationWarning: 'resetCache' deprecated - use 'reset_cache'
  parser.resetCache()
/usr/local/lib/python3.12/dist-packages/matplotlib/_mathtext.py:45: PyparsingDeprecationWarning: 'enablePackrat' deprecated - use 'enable_packrat'
  ParserElement.enablePackrat()
In /usr/local/lib/python3.12/dist-packages/matplotlib/mpl-data/stylelib/classic.mplstyle: 'parseString' deprecated - use 'parse_string'
In /usr/local/lib/python3.12/dist-packages/matplotlib/mpl-data/stylelib/class

In [ ]:
online_features = store.get_online_features(
    features=feature_service,
    entity_rows=[
        {
            "sample_id": 25
        }
    ]
).to_dict()

print(online_features)

{'sample_id': [25], 'missing_skills': [3], 'gap_level_encoded': [2], 'training_hours': [12.0], 'student_score': [0.0], 'industry_score': [7.0], 'alignment_score': [6.0]}


In [ ]:
online_df = pd.DataFrame(
    online_features
)

display(online_df)

,sample_id,missing_skills,gap_level_encoded,training_hours,student_score,industry_score,alignment_score
0,25,3,2,12.0,0.0,7.0,6.0


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
final_model = DecisionTreeClassifier(
    max_depth=4,
    random_state=42
)

final_model.fit(
    X,
    y
)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


DecisionTreeClassifier(max_depth=4, random_state=42)

In [ ]:
X_online = online_df[
    feature_columns
]

prediction = final_model.predict(
    X_online
)

print(
    "Predicted gap level:",
    prediction[0]
)

Predicted gap level: 1


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
gap_labels = {
    0: "Low",
    1: "Medium",
    2: "High"
}

print(
    "Predicted Skill Gap:",
    gap_labels[prediction[0]]
)

Predicted Skill Gap: Medium


In [ ]:
online_features = store.get_online_features(
    features=feature_service,
    entity_rows=[
        {"sample_id": 10},
        {"sample_id": 20},
        {"sample_id": 30},
        {"sample_id": 40}
    ]
).to_dict()

online_df = pd.DataFrame(
    online_features
)

display(online_df)

,sample_id,missing_skills,gap_level_encoded,training_hours,student_score,industry_score,alignment_score
0,10,9,1,5.0,0.0,3.0,3.0
1,20,4,0,10.0,3.0,5.0,1.0
2,30,9,2,21.0,10.0,10.0,8.0
3,40,0,1,19.0,1.0,9.0,4.0


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
online_df["predicted_gap_level"] = (
    final_model.predict(
        online_df[feature_columns]
    )
)

online_df["predicted_gap_level"] = (
    online_df["predicted_gap_level"]
    .map(gap_labels)
)

display(
    online_df[
        [
            "sample_id",
            "predicted_gap_level"
        ]
    ]
)

,sample_id,predicted_gap_level
0,10,Medium
1,20,Medium
2,30,Medium
3,40,Medium


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
